# Команды

## Установка библиотек

- `python3 -m pip install pandas pandas_datareader statsmodels pmdarima`
- `python3 -m pip install matplotlib seaborn`

## Библиотеки

In [ ]:
# Основные библиотеки
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import pandas_datareader.data as web

# Для ARIMA через statsmodels
from statsmodels.tsa.api import ARIMA

# Для ADF и KPSS
from statsmodels.tsa.api import adfuller, kpss

# Для ACF и PACF
from statsmodels.tsa.api import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Для тестов остатков
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

# Для автоматического подбора ARIMA
import pmdarima as pm

# Для графиков
import matplotlib.pyplot as plt
import seaborn as sns

## Загрузка ряда из FRED

In [ ]:
# Импорт данных
y = web.DataReader(
    name="WAAA",
    data_source="fred",
    start="2000-01-01",
    end="2025-12-31"
).dropna()

# Если нужен Series, а не DataFrame
y = y["WAAA"]

## Первая разность ряда

In [ ]:
dy = y.diff().dropna()

## ARIMA заданного порядка без константы/тренда

In [ ]:
mod = ARIMA(
    y,
    order=(1, 0, 3),
    trend="n",
    missing="drop"
)

res = mod.fit()
print(res.summary())

## Получить коэффициенты модели

In [ ]:
print(res.params)

# AR-коэффициенты
phi1 = res.params["ar.L1"]
phi2 = res.params["ar.L2"]

# MA-коэффициенты
theta1 = res.params["ma.L1"]
theta2 = res.params["ma.L2"]


## Округление

In [ ]:
round(phi1, 3)

## Прогноз на несколько периодов

In [ ]:
forecast = res.forecast(steps=2)
print(forecast.round(3))

## ADF-тест

In [ ]:
# Нулевая гипотеза (H0): есть единичный корень

# С константой, без тренда
adf_result = adfuller(
    y,
    regression="c",
    autolag="AIC"
)

# Без константы и без тренда
adf_result = adfuller(
    dy,
    regression="n",
    autolag="AIC"
)

# С константой и трендом:
adf_result = adfuller(
    y,
    regression="ct",
    autolag="AIC"
)

# Достать результаты
adf_stat = adf_result[0]
pvalue = adf_result[1]
used_lag = adf_result[2]
crit_values = adf_result[4]

print(round(adf_stat, 3))
print(round(crit_values["5%"], 3))

# Вывод для ADF
if adf_stat < crit_values["5%"]:
    print("нет единичного корня")
else:
    print("есть единичный корень")


## KPSS-тест

In [ ]:
# Нулевая гипотеза (H0): нет единичного корня, ряд стационарен

# С константой, без тренда
kpss_result = kpss(
    y,
    regression="c",
    nlags="auto"
)

# С константой и трендом
kpss_result = kpss(
    y,
    regression="ct",
    nlags="auto"
)

# Достать результаты
kpss_stat = kpss_result[0]
pvalue = kpss_result[1]
used_lags = kpss_result[2]
crit_values = kpss_result[3]

print(round(kpss_stat, 3))
print(round(crit_values["5%"], 3))

# Вывод для KPSS
if kpss_stat > crit_values["5%"]:
    print("есть единичный корень")
else:
    print("нет единичного корня")


## ACF и PACF

In [ ]:
acf_values = acf(dy, nlags=3)
pacf_values = pacf(dy, nlags=3)

r3 = acf_values[3]
rpart3 = pacf_values[3]

print(round(r3, 3))
print(round(rpart3, 3))

# Графики
plot_acf(dy, lags=20)
plt.show()

plot_pacf(dy, lags=20)
plt.show()

## Ljung-Box тест на серийную корреляцию

In [ ]:
lb_test = acorr_ljungbox(
    res.resid,
    lags=[5],
    return_df=True
)

print(lb_test)

# Достать статистику и p-value
stat = lb_test.loc[5, "lb_stat"]
pvalue = lb_test.loc[5, "lb_pvalue"]

print(round(stat, 3))
print(round(pvalue, 3))

# Вывод
if pvalue < 0.05:
    print("есть серийная корреляция")
else:
    print("нет серийной корреляции")

## ARCH-тест на гетероскедастичность

In [ ]:
lm_stat, lm_pvalue, f_stat, f_pvalue = het_arch(
    res.resid,
    nlags=5
)

print(round(lm_stat, 3))
print(round(lm_pvalue, 3))

# Вывод
if lm_pvalue < 0.05:
    print("есть гетероскедастичность")
else:
    print("нет гетероскедастичности")

## Автоматический подбор ARIMA через `pmdarima`

In [ ]:
arima = pm.auto_arima(
    y,
    test="kpss",
    information_criterion="aic",
    seasonal=False,
    max_p=3,
    max_q=3,
    suppress_warnings=True,
    error_action="ignore",
    trace=True
)

print(arima.order)

# Варианты параметров
test="adf"
test="kpss"

information_criterion="aic"
information_criterion="bic"
information_criterion="aicc"

# Достать порядок
p, d, q = arima.order

print("p =", p)
print("d =", d)
print("q =", q)

## Универсальный шаблон

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas_datareader.data as web
from statsmodels.tsa.api import ARIMA

series = "WAAA"

y = web.DataReader(
    name=series,
    data_source="fred",
    start="2000-01-01",
    end="2025-12-31"
).dropna()

mod = ARIMA(
    y,
    order=(1, 0, 3),
    trend="n",
    missing="drop"
)

res = mod.fit()

print(res.params.round(3))

forecast = res.forecast(steps=2)
print(forecast.round(3))
